# Raw Otsu: development experiment and evaluation

This notebook is the complete development interface for raw Otsu. It verifies the manifest, runs or reloads all 139 development records, evaluates detections against XML annotations, and visualises class performance, runtime, thresholds, fragmentation, and representative images. It never selects validation or test rows.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'data' / 'dataset_split.csv').is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from algorithms.common import load_image
from algorithms.evaluation import parse_voc_boxes
from algorithms.otsu import detect_otsu
from scripts.run_otsu_development import (
    evaluate_records,
    load_development_rows,
    summarise,
    write_results,
)

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 30)

## 1. Development baseline configuration (not frozen)

Development is used to implement and debug this baseline and to identify defensible candidate settings. Final parameter and matching-rule selection belongs to validation. Otsu's intensity threshold is selected automatically for every difference image; it is an output, not a tuned hyperparameter. `IOU_THRESHOLD = 0.50` is only a provisional one-to-one diagnostic rule. The raw pipeline uses no blur, resizing, morphology, contour-area filtering, or region merging.

In [ ]:
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'dataset_split.csv'
RESULTS_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'otsu_development.csv'
SUMMARY_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'otsu_development_summary.json'
BOXES_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / 'otsu_development_boxes'
IOU_THRESHOLD = 0.50
RUN_DEVELOPMENT = False  # Set True to recompute all 139 full-resolution images.

development_baseline = {
    'input': 'aligned reference and defective image pair',
    'shared_preprocessing': 'dimension validation and grayscale conversion',
    'comparison': 'cv2.absdiff',
    'threshold': 'cv2.THRESH_BINARY + cv2.THRESH_OTSU (automatic per image)',
    'contours': 'cv2.RETR_EXTERNAL + cv2.CHAIN_APPROX_SIMPLE',
    'post_processing': 'none',
    'evaluation_iou': IOU_THRESHOLD,
}
pd.Series(development_baseline, name='setting').to_frame()

## 2. Manifest and development-split verification

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)
development_df = manifest_df.loc[manifest_df['split'].eq('development')].copy()

assert len(manifest_df) == 693
assert len(development_df) == 139
assert set(development_df['split']) == {'development'}
assert development_df['image_id'].is_unique

split_counts = manifest_df['split'].value_counts().reindex(
    ['development', 'validation', 'test']
)
class_counts = development_df['defect_class'].value_counts().sort_index()
display(split_counts.rename('images').to_frame())
display(class_counts.rename('development_images').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
split_counts.plot.bar(ax=axes[0], color=['#4C78A8', '#F58518', '#54A24B'])
axes[0].set_title('Authoritative manifest split')
axes[0].set_ylabel('Images')
axes[0].tick_params(axis='x', rotation=0)
class_counts.plot.bar(ax=axes[1], color='#4C78A8')
axes[1].set_title('Development class distribution')
axes[1].set_ylabel('Images')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 3. Execute or reload the development experiment

The existing verified outputs are loaded by default. Set `RUN_DEVELOPMENT = True` to rerun the complete full-resolution experiment. Complete unfiltered predicted boxes are stored in typed NumPy sidecars because raw Otsu produced more than 25 million contours.

In [ ]:
outputs_exist = RESULTS_PATH.is_file() and SUMMARY_PATH.is_file()
if RUN_DEVELOPMENT or not outputs_exist:
    development_rows = load_development_rows(MANIFEST_PATH)
    results = evaluate_records(
        development_rows,
        IOU_THRESHOLD,
        workers=1,
        boxes_directory=BOXES_DIR,
    )
    summary = summarise(results, IOU_THRESHOLD)
    write_results(results, RESULTS_PATH)
    SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    SUMMARY_PATH.write_text(
        json.dumps(summary, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )
    print('Development experiment completed and saved.')
else:
    print('Loading the existing verified development results.')

In [ ]:
results_df = pd.read_csv(RESULTS_PATH)
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

assert len(results_df) == 139
assert set(results_df['split']) == {'development'}
assert set(results_df['status']) == {'success'}
assert summary['records'] == summary['successful'] == 139
assert summary['errors'] == 0
assert np.isclose(summary['iou_threshold'], IOU_THRESHOLD)
assert results_df['predicted_boxes_path'].map(
    lambda path: (PROJECT_ROOT / path).is_file()
).all()

print('Verified result rows:', len(results_df))
print('Errors:', summary['errors'])
results_df.head()

## 4. Overall development evaluation

Object-detection metrics use greedy one-to-one matching at IoU 0.50. True negatives and classification accuracy are not defined for this direct contour-detection output, so precision, recall, F1, matched IoU, false positives, and false negatives are reported.

In [ ]:
overall_metrics = pd.Series(
    summary['overall_box_metrics'], name='raw_otsu'
).to_frame()
runtime_metrics = pd.Series(
    summary['runtime_ms'], name='raw_otsu'
).to_frame()
display(overall_metrics)
display(runtime_metrics)

In [ ]:
class_metrics_df = pd.DataFrame(summary['box_metrics_by_class']).T
class_metrics_df.index.name = 'defect_class'
display(class_metrics_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
class_metrics_df['recall'].plot.bar(ax=axes[0], color='#4C78A8')
axes[0].set_title('Recall by defect class at IoU 0.50')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Recall')
axes[0].tick_params(axis='x', rotation=30)
class_metrics_df[['precision', 'f1_score']].plot.bar(
    ax=axes[1], color=['#E45756', '#72B7B2']
)
axes[1].set_yscale('log')
axes[1].set_title('Precision and F1 by class (log scale)')
axes[1].set_ylabel('Score')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. Otsu thresholds, fragmentation, and runtime

The threshold plot describes Otsu's automatic choices; it is not a manual parameter search. Predicted-region counts use a logarithmic scale because raw differences can generate hundreds of thousands of one-pixel or tiny contours.

In [ ]:
class_order = sorted(results_df['defect_class'].unique())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
results_df.boxplot(
    column='otsu_threshold', by='defect_class', ax=axes[0], rot=30
)
axes[0].set_title('Automatic Otsu threshold')
axes[0].set_xlabel('Defect class')
axes[0].set_ylabel('Intensity threshold')
results_df.boxplot(
    column='predicted_count', by='defect_class', ax=axes[1], rot=30
)
axes[1].set_yscale('symlog', linthresh=1)
axes[1].set_title('Raw predicted regions (log scale)')
axes[1].set_xlabel('Defect class')
axes[1].set_ylabel('External contours')
results_df.boxplot(
    column='processing_time_ms', by='defect_class', ax=axes[2], rot=30
)
axes[2].set_title('Algorithm runtime')
axes[2].set_xlabel('Defect class')
axes[2].set_ylabel('Milliseconds')
fig.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
fragmentation_summary = results_df.groupby('defect_class').agg(
    images=('image_id', 'size'),
    median_threshold=('otsu_threshold', 'median'),
    median_regions=('predicted_count', 'median'),
    maximum_regions=('predicted_count', 'max'),
    mean_runtime_ms=('processing_time_ms', 'mean'),
    recall=('recall', 'mean'),
)
fragmentation_summary

## 6. Per-image results and representative visualisations

In [ ]:
result_columns = [
    'image_id', 'defect_class', 'otsu_threshold', 'predicted_count',
    'ground_truth_count', 'true_positives', 'false_positives',
    'false_negatives', 'precision', 'recall', 'f1_score',
    'mean_matched_iou', 'processing_time_ms',
]
print('Highest matched-IoU development examples')
display(
    results_df.sort_values(
        ['mean_matched_iou', 'recall'], ascending=False
    )[result_columns].head(10)
)
print('Most fragmented development examples')
display(
    results_df.sort_values('predicted_count', ascending=False)[
        result_columns
    ].head(10)
)

In [ ]:
def show_development_result(image_id, maximum_drawn_boxes=500):
    manifest_row = development_df.loc[
        development_df['image_id'].eq(image_id)
    ].iloc[0]
    result_row = results_df.loc[results_df['image_id'].eq(image_id)].iloc[0]
    reference = load_image(PROJECT_ROOT / manifest_row['reference_path'])
    defective = load_image(PROJECT_ROOT / manifest_row['image_path'])
    ground_truth = parse_voc_boxes(
        PROJECT_ROOT / manifest_row['annotation_path']
    )
    detection = detect_otsu(reference, defective)
    overlay = defective.copy()

    for box in ground_truth:
        cv2.rectangle(
            overlay,
            (int(box['xmin']), int(box['ymin'])),
            (int(box['xmax']), int(box['ymax'])),
            (0, 0, 255),
            4,
        )
    boxes_drawn = len(detection.boxes) <= maximum_drawn_boxes
    if boxes_drawn:
        for box in detection.boxes:
            cv2.rectangle(
                overlay,
                (int(box['xmin']), int(box['ymin'])),
                (int(box['xmax']), int(box['ymax'])),
                (0, 255, 0),
                2,
            )

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(cv2.cvtColor(reference, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Reference')
    axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    overlay_note = 'GT red; predictions green' if boxes_drawn else 'GT red; boxes omitted from display'
    axes[1].set_title(overlay_note)
    axes[2].imshow(detection.difference, cmap='gray')
    axes[2].set_title(f'Absolute difference; Otsu={detection.threshold:.0f}')
    axes[3].imshow(detection.mask, cmap='gray')
    axes[3].set_title(f"Raw mask; {int(result_row['predicted_count']):,} contours")
    for axis in axes:
        axis.axis('off')
    fig.suptitle(
        f"{image_id} | class={manifest_row['defect_class']} | "
        f"recall={result_row['recall']:.3f} | "
        f"mean matched IoU={result_row['mean_matched_iou']:.3f}"
    )
    plt.tight_layout()
    plt.show()

    return result_row[result_columns].to_frame(name='value')

In [ ]:
best_image_id = results_df.sort_values(
    ['mean_matched_iou', 'recall'], ascending=False
).iloc[0]['image_id']
show_development_result(best_image_id)

In [ ]:
most_fragmented_image_id = results_df.sort_values(
    'predicted_count', ascending=False
).iloc[0]['image_id']
show_development_result(most_fragmented_image_id)

## 7. Development conclusion

Raw Otsu baseline development is implemented and debugged, but its configuration is not yet validation-frozen and no parameter has been selected to maximise IoU. Otsu chooses its threshold automatically and the experiment intentionally forbids blur, morphology, contour filtering, and merging. The provisional IoU 0.50 rule evaluates localisation; it does not alter detections. Use development findings to define candidate rules, compare those candidates on validation, then freeze one common evaluation rule before running the test split.